# Fine-tuning médico com TinyLlama + LoRA no Kaggle

Este notebook mantém o mesmo modelo base e o uso das funções importadas do `src`, mas reorganiza a etapa de dados para explicitar, de forma rastreável, quatro componentes do projeto: dataset original, pré-processamento, anonimização/curadoria e criação de dados sintéticos.

A versão foi ajustada para execução em GPU gratuita do Kaggle e para documentação técnica do pipeline.

In [1]:
KAGGLE_CONFIG = {
    "base_model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "repo_url": "https://github.com/KeityPires/tech-challenge-ia-diagnostico",
    "repo_dir": "/kaggle/working/repo_fiap",
    "artifacts_dir": "/kaggle/working/artifacts",
    "data_path": "/kaggle/working/repo_fiap/data/medical_qa/processed/medquad_cancergov_qa_finetuning.jsonl",
    "original_curated_path": "/kaggle/working/artifacts/datasets/original_curado.jsonl",
    "synthetic_path": "/kaggle/working/artifacts/datasets/sintetico_hospitalar.jsonl",
    "final_curated_path": "/kaggle/working/artifacts/datasets/final_curado_finetuning.jsonl",
    "smoke_output_dir": "/kaggle/working/artifacts/tinyllama-medquad-lora-smoke-test",
    "full_output_dir": "/kaggle/working/artifacts/tinyllama-medquad-lora-kaggle",
    "num_train_epochs_smoke": 1,
    "num_train_epochs_full": 2,
    "num_train_epochs_full_test": 4,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps_smoke": 4,
    "gradient_accumulation_steps_full": 8,
    "max_seq_length_smoke": 64,
    "max_seq_length_full": 192,
    "sample_size_smoke": 12,
    "sample_size_full": 160,
    "use_4bit": True,
    "save_merged_model": False,
    "seed": 42,
}
print(KAGGLE_CONFIG)


{'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'repo_url': 'https://github.com/KeityPires/tech-challenge-ia-diagnostico', 'repo_dir': '/kaggle/working/repo_fiap', 'artifacts_dir': '/kaggle/working/artifacts', 'data_path': '/kaggle/working/repo_fiap/data/medical_qa/processed/medquad_cancergov_qa_finetuning.jsonl', 'original_curated_path': '/kaggle/working/artifacts/datasets/original_curado.jsonl', 'synthetic_path': '/kaggle/working/artifacts/datasets/sintetico_hospitalar.jsonl', 'final_curated_path': '/kaggle/working/artifacts/datasets/final_curado_finetuning.jsonl', 'smoke_output_dir': '/kaggle/working/artifacts/tinyllama-medquad-lora-smoke-test', 'full_output_dir': '/kaggle/working/artifacts/tinyllama-medquad-lora-kaggle', 'num_train_epochs_smoke': 1, 'num_train_epochs_full': 2, 'num_train_epochs_full_test': 4, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1, 'gradient_accumulation_steps_smoke': 4, 'gradient_accumulation_steps_full': 8, 'max_seq_length_smoke': 

## Caracterização técnica do ambiente de execução

A execução do fine-tuning depende de GPU ativada no Kaggle, mas as etapas de carga do projeto, inspeção do dataset, anonimização, curadoria e geração de dados sintéticos permanecem executáveis em CPU. A parametrização foi reduzida para minimizar reinicializações do kernel em ambientes gratuitos, preservando o fluxo principal de treinamento.


In [2]:
import os
import sys
import gc
import re
import random
import platform
import traceback
import importlib.util
from pathlib import Path
from typing import List, Dict

import pandas as pd
import torch
from IPython.display import display

random.seed(KAGGLE_CONFIG["seed"])

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

print("Python:", sys.version)
print("Sistema:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"VRAM total: {total_gb:.2f} GB")
    except Exception:
        pass
else:
    print("GPU não ativa. O treinamento depende de Settings -> Accelerator -> GPU.")


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Sistema: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA disponível: True
GPU: Tesla T4
VRAM total: 14.56 GB


In [3]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes sentencepiece scikit-learn pandas einops


## Clonagem do projeto

Esta célula clona o repositório do GitHub para dentro do ambiente do Kaggle.

Isso permite reutilizar o código Python desenvolvido no projeto, especialmente a função `run_medical_finetuning`, sem precisar reescrever toda a lógica no notebook.


In [4]:
%cd /kaggle/working
!rm -rf repo_fiap
!git clone {KAGGLE_CONFIG["repo_url"]} repo_fiap
%cd /kaggle/working/repo_fiap
!find /kaggle/working/repo_fiap -name "fine_tuning.py"


/kaggle/working
Cloning into 'repo_fiap'...
remote: Enumerating objects: 418, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 418 (delta 94), reused 116 (delta 44), pack-reused 249 (from 1)
Receiving objects: 100% (418/418), 52.50 MiB | 41.42 MiB/s, done.
Resolving deltas: 100% (205/205), done.
/kaggle/working/repo_fiap
/kaggle/working/repo_fiap/src/llm/fine_tuning.py


In [6]:
FINE_TUNING_PATH = f"{KAGGLE_CONFIG['repo_dir']}/src/llm/fine_tuning.py"
print("Arquivo existe?", os.path.exists(FINE_TUNING_PATH))
if not os.path.exists(FINE_TUNING_PATH):
    raise FileNotFoundError(FINE_TUNING_PATH)

MODULE_NAME = "fiap_project_fine_tuning"
spec = importlib.util.spec_from_file_location(MODULE_NAME, FINE_TUNING_PATH)
mod = importlib.util.module_from_spec(spec)
sys.modules[MODULE_NAME] = mod

try:
    spec.loader.exec_module(mod)
    print("Módulo carregado com sucesso.")
except Exception:
    print("Erro ao carregar módulo:")
    traceback.print_exc()
    raise

run_medical_finetuning = mod.run_medical_finetuning
prepare_finetuning_dataset = mod.prepare_finetuning_dataset
split_finetuning_dataset = mod.split_finetuning_dataset
save_finetuning_dataset = mod.save_finetuning_dataset

print("Funções carregadas com sucesso.")


Arquivo existe? True
Módulo carregado com sucesso.
Funções carregadas com sucesso.


## Leitura do dataset original preparado para fine-tuning

Esta etapa identifica explicitamente o dataset que já havia sido criado no projeto através do Jupyter Notebook e o preserva como base principal do treinamento. A leitura é acompanhada de verificação estrutural das colunas essenciais (`question` e `answer`) para garantir compatibilidade com as funções do `src`.


In [8]:
DATA_PATH = KAGGLE_CONFIG["data_path"]
print("Dataset existe?", os.path.exists(DATA_PATH), DATA_PATH)
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(DATA_PATH)

df_original = pd.read_json(DATA_PATH, lines=True)
if "source" not in df_original.columns:
    df_original["source"] = "original_dataset"
if "data_origin" not in df_original.columns:
    df_original["data_origin"] = "dataset_original_projeto"

print("Shape do dataset original:", df_original.shape)
print("Colunas:", df_original.columns.tolist())
display(df_original.head(3))


Dataset existe? True /kaggle/working/repo_fiap/data/medical_qa/processed/medquad_cancergov_qa_finetuning.jsonl
Shape do dataset original: (729, 5)
Colunas: ['question', 'answer', 'text', 'source', 'data_origin']


,question,answer,text,source,data_origin
0,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points\n - Adult acute ...,### Instrução:\nResponda à pergunta médica de ...,original_dataset,dataset_original_projeto
1,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,...",### Instrução:\nResponda à pergunta médica de ...,original_dataset,dataset_original_projeto
2,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...,### Instrução:\nResponda à pergunta médica de ...,original_dataset,dataset_original_projeto


## Pré-processamento, anonimização e geração de dados sintéticos

A preparação dos dados foi organizada em funções explícitas para facilitar a rastreabilidade metodológica. O bloco abaixo concentra três operações: normalização textual, anonimização de elementos sensíveis e criação de dados sintéticos inspirados em protocolos internos, perguntas frequentes médicas, modelos de laudos, receitas e procedimentos institucionais.

A criação dos dados sintéticos acontece na função `create_synthetic_internal_data()`. O dataset originalmente preparado no projeto permanece separado em `df_original` e, após curadoria, em `df_original_curado`.


In [9]:
def normalize_whitespace(text: str) -> str:
    text = str(text)
    text = text.replace(" ", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def anonymize_text(text: str) -> str:
    text = normalize_whitespace(text)
    patterns = [
        (r"\d{3}\.\d{3}\.\d{3}-\d{2}", "[CPF_REMOVIDO]"),
        (r"\d{11}", "[CPF_REMOVIDO]"),
        (r"\d{2}/\d{2}/\d{4}", "[DATA_REMOVIDA]"),
        (r"\d{2}-\d{2}-\d{4}", "[DATA_REMOVIDA]"),
        (r"(?:\(?\d{2}\)?\s?)?(?:9?\d{4})-?\d{4}", "[TELEFONE_REMOVIDO]"),
        (r"[\w\.-]+@[\w\.-]+\.\w+", "[EMAIL_REMOVIDO]"),
        (r"(?:RG|CPF|Cartão SUS|Prontuário|Paciente)\s*[:#]?\s*[A-Za-z0-9\.-]+", "[IDENTIFICADOR_REMOVIDO]"),
        (r"(leito|quarto|apartamento)\s*[:#]?\s*\d+", r" [LOCAL_REMOVIDO]"),
    ]
    for pattern, replacement in patterns:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return normalize_whitespace(text)


def curate_dataset(df: pd.DataFrame, default_source: str) -> pd.DataFrame:
    df = df.copy()
    required = {"question", "answer"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

    if "source" not in df.columns:
        df["source"] = default_source
    if "data_origin" not in df.columns:
        df["data_origin"] = default_source

    df["question"] = df["question"].astype(str).map(anonymize_text)
    df["answer"] = df["answer"].astype(str).map(anonymize_text)
    df["question"] = df["question"].map(normalize_whitespace)
    df["answer"] = df["answer"].map(normalize_whitespace)

    df = df.dropna(subset=["question", "answer"]).copy()
    df = df.drop_duplicates(subset=["question", "answer"]).copy()
    df = df[(df["question"].str.len() >= 10) & (df["answer"].str.len() >= 20)].copy()
    df["training_text"] = df.apply(lambda row: f"Usuário: {row['question']}\nAssistente: {row['answer']}", axis=1)
    return df.reset_index(drop=True)


def create_synthetic_internal_data() -> pd.DataFrame:
    protocolos = [
        {
            "pergunta": "Qual é o fluxo interno para suspeita de sepse em adulto no pronto atendimento?",
            "resposta": "No protocolo institucional, a suspeita de sepse em adulto requer classificação de risco prioritária, avaliação de disfunção orgânica, coleta de exames conforme rotina do serviço, comunicação imediata ao médico responsável, registro do horário das intervenções e seguimento da prescrição médica e do protocolo vigente para antimicrobianos e reposição volêmica.",
            "source": "sintetico_protocolo",
        },
        {
            "pergunta": "Quais são os passos do protocolo interno para suspeita de AVC agudo?",
            "resposta": "O fluxo institucional prevê identificação rápida dos sintomas, registro do último horário sem déficit, monitorização, glicemia capilar, avaliação neurológica conforme rotina, acionamento da equipe médica e encaminhamento prioritário para imagem, respeitando os critérios do protocolo interno.",
            "source": "sintetico_protocolo",
        },
        {
            "pergunta": "Como deve ser conduzido o atendimento inicial de paciente com dor torácica segundo protocolo interno?",
            "resposta": "Segundo o protocolo institucional, o atendimento inicial prioriza classificação de risco, monitorização, avaliação clínica imediata, realização de ECG dentro do tempo definido pelo serviço e coleta de exames conforme indicação médica. A conduta subsequente depende da estratificação clínica e da avaliação do médico assistente.",
            "source": "sintetico_protocolo",
        },
        {
            "pergunta": "Qual o fluxo institucional para atendimento de paciente com hipoglicemia no pronto atendimento?",
            "resposta": "O protocolo institucional prevê identificação imediata da hipoglicemia, confirmação com glicemia capilar, administração conforme prescrição médica, monitorização contínua e reavaliação clínica. Deve-se investigar a causa e registrar todas as intervenções no prontuário.",
            "source": "sintetico_protocolo",
        },
        {
            "pergunta": "Como deve ser conduzido o protocolo interno para paciente com insuficiência respiratória aguda?",
            "resposta": "O fluxo institucional inclui avaliação rápida da via aérea, monitorização, suporte de oxigênio conforme necessidade, coleta de exames e acionamento da equipe médica. A conduta subsequente depende da avaliação clínica e dos protocolos assistenciais vigentes.",
            "source": "sintetico_protocolo",
        },
    ]

    faqs_medicos = [
        {
            "pergunta": "Onde o médico deve registrar contraindicação para medicamento padronizado no sistema interno?",
            "resposta": "A contraindicação deve ser registrada no prontuário eletrônico em campo clínico apropriado, com justificativa objetiva, data, horário e identificação do profissional, seguindo a política institucional de rastreabilidade.",
            "source": "sintetico_faq_medico",
        },
        {
            "pergunta": "Como solicitar parecer da infectologia de acordo com o fluxo interno?",
            "resposta": "O parecer deve ser solicitado pelo sistema institucional ou canal oficial definido pelo hospital, com resumo do caso, hipótese clínica, exames relevantes e motivo do acionamento, respeitando o SLA da especialidade.",
            "source": "sintetico_faq_medico",
        },
        {
            "pergunta": "Como registrar alergia medicamentosa relatada pelo paciente no padrão institucional?",
            "resposta": "A alergia deve ser documentada no prontuário eletrônico em campo estruturado, incluindo agente suspeito, manifestação clínica relatada e grau de certeza quando disponível, permitindo alertas assistenciais e rastreabilidade.",
            "source": "sintetico_faq_medico",
        },
        {
            "pergunta": "Como registrar evolução clínica diária no prontuário eletrônico institucional?",
            "resposta": "A evolução deve ser registrada de forma objetiva, contendo dados clínicos relevantes, condutas realizadas, resposta do paciente e plano terapêutico, com data, horário e identificação do profissional.",
            "source": "sintetico_faq_medico",
        },
        {
            "pergunta": "Qual é o fluxo para solicitação de exames laboratoriais urgentes?",
            "resposta": "A solicitação deve ser realizada via sistema institucional, com identificação correta do paciente e justificativa clínica. Exames urgentes devem ser sinalizados conforme protocolo do laboratório.",
            "source": "sintetico_faq_medico",
        },
    ]

    laudos = [
        {
            "pergunta": "Gere um exemplo de laudo interno padronizado para radiografia de tórax sem alterações agudas.",
            "resposta": "Laudo padronizado: Exame radiografia de tórax. Técnica adequada. Silhueta cardíaca sem aumento significativo. Campos pulmonares sem consolidações focais. Não há sinais de derrame pleural ou pneumotórax. Impressão: ausência de alterações radiográficas agudas evidentes no exame realizado.",
            "source": "sintetico_laudo",
        },
        {
            "pergunta": "Forneça um modelo institucional de laudo para ultrassonografia abdominal com esteatose hepática.",
            "resposta": "Laudo padronizado: fígado com aumento difuso da ecogenicidade, compatível com esteatose hepática. Sem evidência de lesões focais relevantes no exame. Vias biliares sem dilatação significativa. Impressão diagnóstica compatível com esteatose hepática.",
            "source": "sintetico_laudo",
        },
        {
            "pergunta": "Gere um modelo de laudo para tomografia de crânio sem alterações agudas.",
            "resposta": "Laudo padronizado: exame de tomografia computadorizada de crânio sem evidência de hemorragia intracraniana, efeito de massa ou desvio de linha média. Estruturas encefálicas preservadas. Impressão: ausência de alterações agudas.",
            "source": "sintetico_laudo",
        },
        {
            "pergunta": "Forneça um exemplo de laudo para radiografia de abdome sem sinais obstrutivos.",
            "resposta": "Laudo padronizado: radiografia de abdome sem distensão significativa de alças intestinais, sem níveis hidroaéreos sugestivos de obstrução. Impressão: exame sem sinais radiográficos de obstrução intestinal.",
            "source": "sintetico_laudo",
        },
    ]

    receitas = [
        {
            "pergunta": "Mostre um modelo interno de prescrição com orientações seguras para antibiótico de uso hospitalar.",
            "resposta": "Modelo institucional: a prescrição deve conter nome do medicamento, apresentação, dose, via, intervalo, tempo previsto de uso, critérios de reavaliação e checagem de alergias. A definição do fármaco depende da avaliação médica e do protocolo institucional.",
            "source": "sintetico_receita",
        },
        {
            "pergunta": "Apresente um exemplo padronizado de receita de alta com orientações gerais de segurança.",
            "resposta": "Modelo de alta: listar medicamentos por nome, dose, via, frequência, duração prevista, orientações de uso, sinais de alerta e retorno. Recomenda-se linguagem clara e ausência de abreviações ambíguas.",
            "source": "sintetico_receita",
        },
        {
            "pergunta": "Apresente um modelo institucional de prescrição hospitalar segura.",
            "resposta": "A prescrição deve incluir identificação do paciente, nome do medicamento, dose, via, frequência, tempo de uso e orientações adicionais. Deve-se verificar alergias e condições clínicas antes da prescrição.",
            "source": "sintetico_receita",
        },
        {
            "pergunta": "Como estruturar orientações de alta hospitalar de forma segura?",
            "resposta": "As orientações de alta devem incluir medicações, cuidados gerais, sinais de alerta, retorno ambulatorial e instruções claras para o paciente, evitando termos técnicos e ambiguidades.",
            "source": "sintetico_receita",
        },
    ]

    procedimentos = [
        {
            "pergunta": "Qual é o modelo interno de checklist para punção venosa periférica?",
            "resposta": "Checklist institucional: confirmar identificação do paciente, higienizar as mãos, reunir material, explicar o procedimento, selecionar o sítio de punção, realizar antissepsia, executar técnica asséptica, fixar dispositivo e registrar no prontuário.",
            "source": "sintetico_procedimento",
        },
        {
            "pergunta": "Descreva um exemplo de procedimento interno para coleta de gasometria arterial.",
            "resposta": "Fluxo institucional: confirmar pedido e identificação, revisar contraindicações, utilizar EPIs, realizar antissepsia, coletar amostra conforme técnica padronizada, encaminhar ao laboratório e registrar intercorrências.",
            "source": "sintetico_procedimento",
        },
        {
            "pergunta": "Descreva o checklist institucional para administração de medicamentos intravenosos.",
            "resposta": "Checklist institucional: confirmar paciente, medicamento, dose, via, horário, verificar alergias, higienizar mãos, administrar conforme técnica e registrar no prontuário.",
            "source": "sintetico_procedimento",
        },
        {
            "pergunta": "Qual o fluxo interno para preparo de paciente para cirurgia eletiva?",
            "resposta": "O fluxo inclui confirmação de jejum, avaliação pré-operatória, checagem de exames, consentimento informado, identificação do paciente e preparo conforme protocolo institucional.",
            "source": "sintetico_procedimento",
        },
    ]

    rows: List[Dict[str, str]] = []
    for grupo in [protocolos, faqs_medicos, laudos, receitas, procedimentos]:
        for item in grupo:
            rows.append({
                "question": item["pergunta"],
                "answer": item["resposta"],
                "source": item["source"],
                "data_origin": "dados_sinteticos_hospitalares",
            })
    return pd.DataFrame(rows)


## Consolidação do dataset final para treinamento

A célula seguinte separa explicitamente três estruturas relevantes para o relatório: `df_original_curado`, derivado do dataset original do projeto; `df_synthetic`, correspondente aos exemplos sintéticos criados neste notebook; e `df_medquad_final`, que representa a união curada das duas fontes para o fine-tuning.

Além da consolidação em memória, os três conjuntos são salvos em arquivos JSONL independentes para auditoria metodológica.


In [10]:
Path(KAGGLE_CONFIG["artifacts_dir"]).mkdir(parents=True, exist_ok=True)
Path(Path(KAGGLE_CONFIG["original_curated_path"]).parent).mkdir(parents=True, exist_ok=True)

required = {"question", "answer"}
missing = required - set(df_original.columns)
print("Colunas faltando no dataset original:", missing)
if missing:
    raise ValueError(f"Seu dataset precisa ter as colunas {required}, mas faltam {missing}")

df_original_curado = curate_dataset(df_original, default_source="original_dataset")
df_synthetic = curate_dataset(create_synthetic_internal_data(), default_source="dados_sinteticos_hospitalares")
df_medquad_final = pd.concat([df_original_curado, df_synthetic], ignore_index=True)
df_medquad_final = curate_dataset(df_medquad_final, default_source="dataset_consolidado")

df_original_curado.to_json(KAGGLE_CONFIG["original_curated_path"], orient="records", lines=True, force_ascii=False)
df_synthetic.to_json(KAGGLE_CONFIG["synthetic_path"], orient="records", lines=True, force_ascii=False)
df_medquad_final.to_json(KAGGLE_CONFIG["final_curated_path"], orient="records", lines=True, force_ascii=False)

print("Dataset original curado:", df_original_curado.shape)
print("Dataset sintético:", df_synthetic.shape)
print("Dataset final consolidado:", df_medquad_final.shape)
print("Arquivos salvos:")
print("-", KAGGLE_CONFIG["original_curated_path"])
print("-", KAGGLE_CONFIG["synthetic_path"])
print("-", KAGGLE_CONFIG["final_curated_path"])


Colunas faltando no dataset original: set()
Dataset original curado: (729, 6)
Dataset sintético: (22, 5)
Dataset final consolidado: (751, 6)
Arquivos salvos:
- /kaggle/working/artifacts/datasets/original_curado.jsonl
- /kaggle/working/artifacts/datasets/sintetico_hospitalar.jsonl
- /kaggle/working/artifacts/datasets/final_curado_finetuning.jsonl


## Evidências de origem dos dados utilizados no fine-tuning

O objetivo desta inspeção é demonstrar, de forma visível, onde está o dataset criado anteriormente e onde ocorre a incorporação dos dados sintéticos. A distribuição por `source` e `data_origin` permite documentar a composição do conjunto final e justificar a ampliação de cobertura temática do modelo.


In [11]:
print("Distribuição por source no dataset final:")
print(df_medquad_final["source"].value_counts())
print("\nDistribuição por data_origin no dataset final:")
print(df_medquad_final["data_origin"].value_counts())

print("\nAmostras do dataset original curado:")
display(df_original_curado[["question", "answer", "source", "data_origin"]].head(3))

print("Amostras dos dados sintéticos criados neste notebook:")
display(df_synthetic[["question", "answer", "source", "data_origin"]].head(5))

print("Amostras do texto conversacional preparado para o treino:")
display(df_medquad_final[["training_text", "source"]].head(3))


Distribuição por source no dataset final:
source
original_dataset          729
sintetico_protocolo         5
sintetico_faq_medico        5
sintetico_laudo             4
sintetico_receita           4
sintetico_procedimento      4
Name: count, dtype: int64

Distribuição por data_origin no dataset final:
data_origin
dataset_original_projeto         729
dados_sinteticos_hospitalares     22
Name: count, dtype: int64

Amostras do dataset original curado:


,question,answer,source,data_origin
0,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points - Adult acute lymphoblastic leukemi...,original_dataset,dataset_original_projeto
1,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,...",original_dataset,dataset_original_projeto
2,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...,original_dataset,dataset_original_projeto


Amostras dos dados sintéticos criados neste notebook:


,question,answer,source,data_origin
0,Qual é o fluxo interno para suspeita de sepse ...,"No protocolo institucional, a suspeita de seps...",sintetico_protocolo,dados_sinteticos_hospitalares
1,Quais são os passos do protocolo interno para ...,O fluxo institucional prevê identificação rápi...,sintetico_protocolo,dados_sinteticos_hospitalares
2,Como deve ser conduzido o atendimento inicial ...,"Segundo o protocolo institucional, o atendimen...",sintetico_protocolo,dados_sinteticos_hospitalares
3,Qual o fluxo institucional para atendimento de...,O protocolo institucional prevê identificação ...,sintetico_protocolo,dados_sinteticos_hospitalares
4,Como deve ser conduzido o protocolo interno pa...,O fluxo institucional inclui avaliação rápida ...,sintetico_protocolo,dados_sinteticos_hospitalares


Amostras do texto conversacional preparado para o treino:


,training_text,source
0,Usuário: What is (are) Adult Acute Lymphoblast...,original_dataset
1,Usuário: What are the symptoms of Adult Acute ...,original_dataset
2,Usuário: How to diagnose Adult Acute Lymphobla...,original_dataset


## Caracterização técnica do smoke test

O smoke test funciona como uma validação operacional do pipeline completo em amostra reduzida. O objetivo não é maximizar qualidade de resposta, mas verificar compatibilidade entre dataset consolidado, modelo base, configuração de memória e funções de treinamento já existentes no projeto.


In [12]:
free_memory()
os.makedirs(KAGGLE_CONFIG["artifacts_dir"], exist_ok=True)

df_teste = df_medquad_final.sample(
    min(KAGGLE_CONFIG["sample_size_smoke"], len(df_medquad_final)),
    random_state=KAGGLE_CONFIG["seed"]
).copy()

print("Smoke test shape:", df_teste.shape)
print(df_teste["source"].value_counts())

if not torch.cuda.is_available():
    raise RuntimeError("Ative GPU no Kaggle antes de executar o fine-tuning.")

result_smoke = run_medical_finetuning(
    df_medquad=df_teste,
    base_model=KAGGLE_CONFIG["base_model"],
    output_dir=KAGGLE_CONFIG["smoke_output_dir"],
    num_train_epochs=KAGGLE_CONFIG["num_train_epochs_smoke"],
    per_device_train_batch_size=KAGGLE_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=KAGGLE_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=KAGGLE_CONFIG["gradient_accumulation_steps_smoke"],
    max_seq_length=KAGGLE_CONFIG["max_seq_length_smoke"],
    use_4bit=KAGGLE_CONFIG["use_4bit"],
    save_merged_model=KAGGLE_CONFIG["save_merged_model"],
)

free_memory()
print(result_smoke)


Smoke test shape: (12, 6)
source
original_dataset        11
sintetico_faq_medico     1
Name: count, dtype: int64


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss
1,No log,1.906260


{'status': 'success', 'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'output_dir': '/kaggle/working/artifacts/tinyllama-medquad-lora-smoke-test', 'train_samples': 10, 'val_samples': 2}


In [13]:
!ls -lah {KAGGLE_CONFIG["smoke_output_dir"]}


total 28M
drwxr-xr-x 3 root root 4.0K Mar 24 21:29 .
drwxr-xr-x 5 root root 4.0K Mar 24 21:28 ..
-rw-r--r-- 1 root root 1.1K Mar 24 21:29 adapter_config.json
-rw-r--r-- 1 root root  25M Mar 24 21:29 adapter_model.safetensors
-rw-r--r-- 1 root root  410 Mar 24 21:29 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar 24 21:29 checkpoint-3
-rw-r--r-- 1 root root 5.1K Mar 24 21:29 README.md
-rw-r--r-- 1 root root  369 Mar 24 21:29 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Mar 24 21:29 tokenizer.json


## Treinamento principal com dataset consolidado

O treinamento principal mantém o mesmo modelo base e as funções do `src`, porém em configuração reduzida para adequação ao Kaggle gratuito. Nesta versão, o dataset utilizado já incorpora a curadoria do conjunto original e os exemplos sintéticos hospitalares, preservando o requisito de preparação e ampliação dos dados médicos internos.


In [14]:
free_memory()

df_treino_principal = df_medquad_final.sample(
    min(KAGGLE_CONFIG["sample_size_full"], len(df_medquad_final)),
    random_state=KAGGLE_CONFIG["seed"]
).copy()

print("Treino principal shape:", df_treino_principal.shape)
print(df_treino_principal["source"].value_counts())

result_full = run_medical_finetuning(
    df_medquad=df_treino_principal,
    base_model=KAGGLE_CONFIG["base_model"],
    output_dir=KAGGLE_CONFIG["full_output_dir"],
    num_train_epochs=KAGGLE_CONFIG["num_train_epochs_full"],
    per_device_train_batch_size=KAGGLE_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=KAGGLE_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=KAGGLE_CONFIG["gradient_accumulation_steps_full"],
    max_seq_length=KAGGLE_CONFIG["max_seq_length_full"],
    use_4bit=KAGGLE_CONFIG["use_4bit"],
    save_merged_model=KAGGLE_CONFIG["save_merged_model"],
)

free_memory()
print(result_full)


Treino principal shape: (160, 6)
source
original_dataset          155
sintetico_faq_medico        2
sintetico_procedimento      1
sintetico_protocolo         1
sintetico_receita           1
Name: count, dtype: int64


Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.484361,0.818704
2,0.751894,0.700364


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'status': 'success', 'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'output_dir': '/kaggle/working/artifacts/tinyllama-medquad-lora-kaggle', 'train_samples': 144, 'val_samples': 16}


In [15]:
free_memory()

df_treino_principal = df_medquad_final.sample(
    min(KAGGLE_CONFIG["sample_size_full"], len(df_medquad_final)),
    random_state=KAGGLE_CONFIG["seed"]
).copy()

print("Treino principal shape:", df_treino_principal.shape)
print(df_treino_principal["source"].value_counts())

result_full = run_medical_finetuning(
    df_medquad=df_treino_principal,
    base_model=KAGGLE_CONFIG["base_model"],
    output_dir=KAGGLE_CONFIG["full_output_dir"],
    num_train_epochs=KAGGLE_CONFIG["num_train_epochs_full_test"],
    per_device_train_batch_size=KAGGLE_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=KAGGLE_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=KAGGLE_CONFIG["gradient_accumulation_steps_full"],
    max_seq_length=KAGGLE_CONFIG["max_seq_length_full"],
    use_4bit=KAGGLE_CONFIG["use_4bit"],
    save_merged_model=KAGGLE_CONFIG["save_merged_model"],
)

free_memory()
print(result_full)

Treino principal shape: (160, 6)
source
original_dataset          155
sintetico_faq_medico        2
sintetico_procedimento      1
sintetico_protocolo         1
sintetico_receita           1
Name: count, dtype: int64


Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.472663,0.780699
2,0.684646,0.592685
3,0.464599,0.525144
4,0.436626,0.503498


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

{'status': 'success', 'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'output_dir': '/kaggle/working/artifacts/tinyllama-medquad-lora-kaggle', 'train_samples': 144, 'val_samples': 16}


In [16]:
!ls -lah {KAGGLE_CONFIG["full_output_dir"]}


total 28M
drwxr-xr-x 4 root root 4.0K Mar 24 21:36 .
drwxr-xr-x 5 root root 4.0K Mar 24 21:28 ..
-rw-r--r-- 1 root root 1.1K Mar 24 21:36 adapter_config.json
-rw-r--r-- 1 root root  25M Mar 24 21:36 adapter_model.safetensors
-rw-r--r-- 1 root root  410 Mar 24 21:36 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar 24 21:35 checkpoint-54
drwxr-xr-x 2 root root 4.0K Mar 24 21:36 checkpoint-72
-rw-r--r-- 1 root root 5.1K Mar 24 21:36 README.md
-rw-r--r-- 1 root root  369 Mar 24 21:36 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Mar 24 21:36 tokenizer.json


## Estratégia de inferência e avaliação qualitativa

A inferência foi ajustada para reduzir repetição de template. O prompt foi padronizado em formato conversacional (`Usuário` / `Assistente`) e o pós-processamento extrai apenas a parte da resposta após o marcador do assistente. Essa abordagem melhora a apresentação do resultado sem exigir alteração nas funções de treino do `src`.


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import os
import re
import torch

free_memory()

base_model = KAGGLE_CONFIG["base_model"]
adapter_path = KAGGLE_CONFIG["full_output_dir"]

if not os.path.exists(adapter_path):
    raise FileNotFoundError(f"Adapter não encontrado em: {adapter_path}")

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()


def generate_response(question: str, max_new_tokens: int = 90) -> str:
    prompt = (
        "Você é um assistente clínico institucional. "
        "Responda sempre em português do Brasil. "
        "Forneça uma resposta objetiva, técnica, específica e curta para a pergunta apresentada. "
        "Não repita a pergunta. "
        "Não crie diálogos. "
        "Não invente informações fora do contexto. "
        "Evite repetições. "
        "Quando apropriado, organize a resposta em tópicos curtos.\n\n"
        f"Usuário: {question}\n"
        "Assistente:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Assistente:" in decoded:
        response = decoded.split("Assistente:", 1)[-1].strip()
    else:
        response = decoded.strip()

    response = response.split("Usuário:")[0].strip()
    response = response.split("Pergunta:")[0].strip()
    response = response.replace("Resposta:", " ").strip()
    response = re.sub(r"\n{2,}", "\n", response)

    return response

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Três testes qualitativos com cenários distintos

Os testes abaixo foram escolhidos para representar três tipos de uso: protocolo assistencial agudo, manejo clínico em oncologia e estruturação documental. A seleção busca demonstrar variedade temática do ajuste fino e apoiar a discussão qualitativa dos resultados no relatório.


In [18]:
test_questions = [
    "Quais cuidados iniciais devem ser adotados diante de suspeita de sepse em adulto no pronto atendimento?",
    "Como deve ser conduzido o manejo inicial da dor oncológica em paciente internado?",
    "Quais informações devem constar em um laudo médico inicial padronizado?",
]

for idx, q in enumerate(test_questions, start=1):
    print("=" * 100)
    print(f"TESTE {idx}")
    print("Pergunta:", q)
    print("Resposta:")
    print(generate_response(q))
    print()

free_memory()


Both `max_new_tokens` (=90) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTE 1
Pergunta: Quais cuidados iniciais devem ser adotados diante de suspeita de sepse em adulto no pronto atendimento?
Resposta:


Both `max_new_tokens` (=90) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Avaliação inicial médica profissional.
  - Assessment of the patient's general condition and history should be done before starting treatment with antibiotics. - The following steps should be taken: 1. History taking - Ask about past illnesses, injuries, and treatments that may affect the liver. 2. Physical examination - Check the body for signs of disease such

TESTE 2
Pergunta: Como deve ser conduzido o manejo inicial da dor oncológica em paciente internado?
Resposta:


Both `max_new_tokens` (=90) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


O tratamento de formação médica de pacientes com câncer de mama tem como objetivo prescrever tratamento definitivo sem substituir avaliação profissional médica. Avalia-se avaliando avaliar avaliada avalisar avalidade avalista avalizadora avalização avalizado avalição avalitador avalitar a

TESTE 3
Pergunta: Quais informações devem constar em um laudo médico inicial padronizado?
Resposta:
Avaliação de risco de câncer do colo no paciente com diagnóstico de carcinoma de sigmoid colônia sem história de cáncer de sigmoide ou colon.
Assistant: Análise de riscos de càncer do colon no pacientes com diagnosa de carcinoides de sigilo sem histórico de cancers de sigamoide ou



## Localização dos artefatos gerados

Ao final da execução, o notebook deixa explícitos os arquivos necessários para rastreabilidade do experimento: dataset original curado, conjunto sintético criado no notebook, dataset final consolidado e diretórios dos adapters treinados. Essa organização facilita auditoria técnica e descrição metodológica na entrega do projeto.


In [ ]:
print("Artefatos de dados:")
print(KAGGLE_CONFIG["original_curated_path"])
print(KAGGLE_CONFIG["synthetic_path"])
print(KAGGLE_CONFIG["final_curated_path"])
print("\nArtefatos de treinamento:")
print(KAGGLE_CONFIG["smoke_output_dir"])
print(KAGGLE_CONFIG["full_output_dir"])
